# 01 – Ingest policies
Reads the policy documents from `data/policies`, splits them into one chunk per `##` section and saves the chunks to a Delta table that the Vector Search index will be built on.

In [0]:
CATALOG = "workspace"
SCHEMA = "policy_rag"
CHUNKS_TABLE = f"{CATALOG}.{SCHEMA}.policy_chunks"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

In [0]:
import os

print("Working directory:", os.getcwd())
print("Notebook path:", dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get())
print("Contents of ..:", os.listdir(".."))

In [0]:
from pathlib import Path

POLICY_DIR = Path("../data/policies").resolve()
policy_files = sorted(POLICY_DIR.glob("*.md"))

print(POLICY_DIR)
for f in policy_files:
    print(" -", f.name)

In [0]:
def parse_policy(path):
    """Split a policy file into one chunk per ## section, with metadata from the frontmatter."""
    text = path.read_text(encoding="utf-8")

    # The frontmatter is the block between the first two "---" lines
    _, frontmatter, body = text.split("---", 2)
    meta = {}
    for line in frontmatter.strip().splitlines():
        key, value = line.split(":", 1)
        meta[key.strip()] = value.strip()

    # Group lines under their ## heading. Text before the first ## becomes "Overview"
    sections = []
    heading, lines = "Overview", []
    for line in body.strip().splitlines():
        if line.startswith("# "):
            continue  # skip the document title, it is already in the metadata
        if line.startswith("## "):
            sections.append((heading, lines))
            heading, lines = line[3:].strip(), []
        else:
            lines.append(line)
    sections.append((heading, lines))

    chunks = []
    for i, (heading, lines) in enumerate(sections):
        content = "\n".join(lines).strip()
        if not content:
            continue
        chunks.append({
            "chunk_id": f"{path.stem}__{i:02d}",
            "document": meta["document"],
            "section": heading,
            "source_file": path.name,
            "version": meta["version"],
            "owner": meta["owner"],
            # Document and section name are put in the text so each chunk makes sense on its own
            "text": f"{meta['document']} - {heading}\n\n{content}",
            "n_words": len(content.split()),
        })
    return chunks

In [0]:
all_chunks = []
for f in policy_files:
    chunks = parse_policy(f)
    all_chunks.extend(chunks)
    print(f"{f.name}: {len(chunks)} chunks")

print("Total:", len(all_chunks))
print()
print(all_chunks[0]["text"])

In [0]:
df = spark.createDataFrame(all_chunks)

(df.write
   .mode("overwrite")
   .option("overwriteSchema", "true")
   .saveAsTable(CHUNKS_TABLE))

# Vector Search needs Change Data Feed to keep the index in sync with the table
spark.sql(f"ALTER TABLE {CHUNKS_TABLE} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

display(spark.table(CHUNKS_TABLE))